In [ ]:
# LOCATION : https://github.com/purnasai/Dino_V2
import torch
from torchvision import models, transforms
import cv2
import os
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from PIL import Image

os.environ["XFORMERS_DISABLED"] = "1" # Switch to enable xFormers
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

/home/osero/miniconda3/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load DINO ViT model from torchvision (for example, ViT small or base model trained with DINO)
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14').to(device)
model.eval()

# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main


In [15]:
import cv2
import os
import numpy as np

# Step 2: Function to Extract Frames from Video
def extract_frames(video_path, frame_rate=5):
    """Extract frames from a video at a specified frame rate."""
    video_capture = cv2.VideoCapture(video_path)

    # Get the total number of frames
    total_frames = int(video_capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_numbers = [i for i in range(0, total_frames, frame_rate)]
    frames = []

    for frame_number in frame_numbers:
            
        # Set the video capture to requested frame number
        video_capture.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

        # Read the frame
        success, img = video_capture.read()
        if success:
            frames.append(img)
    return frames


# # Step 2: Function to Extract Frames from Video
# def extract_frames(video_path, frame_rate=5):
#     """Extract frames from a video at a specified frame rate."""
#     frames = []
#     video_capture = cv2.VideoCapture(video_path)
#     fps = video_capture.get(cv2.CAP_PROP_FPS)
#     interval = int(fps / frame_rate)
    
#     frame_count = 0
#     while video_capture.isOpened():
#         ret, frame = video_capture.read()
#         if not ret:
#             break
#         if frame_count % interval == 0:
#             # Convert BGR (OpenCV default) to RGB
#             frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             frames.append(frame)
#         frame_count += 1

#     video_capture.release()
#     return frames


In [24]:
# Step 3: Function to Extract Embeddings for a List of Frames
def extract_video_embedding(frames):
    """Extracts and averages embeddings for a list of frames."""
    embeddings = []
    with torch.no_grad():
        for frame in frames:
            # Convert frame to PIL image and apply transformations
            frame = Image.fromarray(frame)
            input_tensor = transform(frame).unsqueeze(0).to(device)  # Add batch dimension
            features = model(input_tensor)
            embeddings.append(features.squeeze().cpu().numpy())
    # Average the embeddings to get a single representation for the video
    video_embedding = np.mean(embeddings, axis=0)
    return video_embedding

In [ ]:

# Step 4: Process All Videos in the Dataset
# Set paths to your video dataset and labels
video_folder = "/media/osero/SamsungSSD/CMPE_SSD/videos"
video_labels = []  # Populate this with the corresponding labels for each video
video_embeddings = []

for label_folder in os.listdir(video_folder):
    full_label_folder = os.path.join(video_folder, label_folder)
    label = int(label_folder)
    for video_file in os.listdir(full_label_folder):
        if video_file.endswith(".mp4") or video_file.endswith(".avi"):  # Add other formats if needed
            video_path = os.path.join(full_label_folder, video_file)
            # Extract frames and then embeddings for each video
            frames = extract_frames(video_path, frame_rate=5)  # Adjust frame rate as needed
            video_embedding = extract_video_embedding(frames)
            video_embeddings.append(video_embedding)
            
            # Append label (this assumes you have a list of labels for each video)
            # Make sure video_labels has a corresponding entry for each video
            # label = ... # Set the appropriate label for this video
            video_labels.append(label)


abc = 4

KeyboardInterrupt: 

In [67]:
# Step 5: Train a Classifier on the Video Embeddings
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(video_embeddings, video_labels, test_size=0.2)

# Initialize a k-NN classifier
knn = KNeighborsClassifier(n_neighbors=5)

# Train the classifier
knn.fit(X_train, y_train)

KNeighborsClassifier()

In [69]:
# Predict on the test set and calculate accuracy
y_pred = knn.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 12.52%


In [ ]:
import pickle

with open('features.pickle', 'wb') as handle:
    pickle.dump((video_embeddings, video_labels), handle, protocol=pickle.HIGHEST_PROTOCOL)

np.unique(video_labels)

array([ 18,  22,  28,  46,  48,  53,  58,  61,  66,  68,  83,  86,  98,
       117, 118, 131, 147, 148, 152, 158, 159, 164, 165, 169, 193, 198,
       199, 211, 212, 223, 244, 251, 263, 266, 288, 295, 307, 323, 334,
       338, 342, 356, 374, 378, 385, 393, 400, 435, 443, 444, 445, 467,
       473, 483, 489, 492, 493, 507, 529, 537, 545, 549, 558, 580, 586,
       594, 596, 604, 610, 616, 625, 631, 632, 643, 664, 667, 692, 699,
       712, 721, 726, 735])